In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
import sklearn

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [6]:
#The first step is to inspect which variables should be used as features for prediction and which variables should be left out.

credit_fraud = pd.read_csv('../data/credit_card_fraud_dataset.csv')
credit_fraud.head()

,TransactionID,TransactionDate,Amount,MerchantID,TransactionType,Location,IsFraud
0,1,2024-04-03 14:15:35.462794,4189.27,688,refund,San Antonio,0
1,2,2024-03-19 13:20:35.462824,2659.71,109,refund,Dallas,0
2,3,2024-01-08 10:08:35.462834,784.00,394,purchase,New York,0
3,4,2024-04-13 23:50:35.462850,3514.40,944,purchase,Philadelphia,0
4,5,2024-07-12 18:51:35.462858,369.07,475,purchase,Phoenix,0


In [7]:
credit_fraud.shape

(100000, 7)

In [8]:
credit_fraud.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   TransactionID    100000 non-null  int64  
 1   TransactionDate  100000 non-null  str    
 2   Amount           100000 non-null  float64
 3   MerchantID       100000 non-null  int64  
 4   TransactionType  100000 non-null  str    
 5   Location         100000 non-null  str    
 6   IsFraud          100000 non-null  int64  
dtypes: float64(1), int64(3), str(3)
memory usage: 5.3 MB


In [9]:
credit_fraud.nunique().sort_values()

IsFraud                 2
TransactionType         2
Location               10
MerchantID           1000
Amount              90621
TransactionDate    100000
TransactionID      100000
dtype: int64

In [10]:
#Now before we choose the features, we need to know how the categorical and target variables look like.

credit_fraud['TransactionType'].value_counts()

TransactionType
refund      50131
purchase    49869
Name: count, dtype: int64

In [11]:
credit_fraud['Location'].value_counts()

Location
Chicago         10193
San Diego       10111
Dallas          10076
San Antonio     10062
New York         9993
Houston          9991
Phoenix          9960
Los Angeles      9936
Philadelphia     9873
San Jose         9805
Name: count, dtype: int64

In [12]:
credit_fraud['IsFraud'].value_counts(normalize=True)*100

IsFraud
0    99.0
1     1.0
Name: proportion, dtype: float64

In [13]:
credit_fraud.groupby('MerchantID')['IsFraud'].agg(['count', 'sum', 'mean']).sort_values('mean', ascending=False).head(10)

,count,sum,mean
MerchantID,,,
640,86,5,0.058140
156,103,5,0.048544
583,106,5,0.047170
659,88,4,0.045455
939,91,4,0.043956
436,96,4,0.041667
568,97,4,0.041237
968,73,3,0.041096
401,98,4,0.040816


Conclusion: For the features, we will leave out merchant ID and TransactionID from the features, including the target variable IsFraud.

In [14]:
#Now let's work on transaction date and time. We will convert the transaction date and time into a datetime object and then extract the hour, day, month, and year from it.

credit_fraud['TransactionDate'] = pd.to_datetime(credit_fraud['TransactionDate'])
credit_fraud['TransactionHour'] = credit_fraud['TransactionDate'].dt.hour
credit_fraud['TransactionDay'] = credit_fraud['TransactionDate'].dt.day_name()
credit_fraud['TransactionMonth'] = credit_fraud['TransactionDate'].dt.month_name()
credit_fraud['TransactionYear'] = credit_fraud['TransactionDate'].dt.year

credit_fraud.head()

,TransactionID,TransactionDate,Amount,MerchantID,TransactionType,Location,IsFraud,TransactionHour,TransactionDay,TransactionMonth,TransactionYear
0,1,2024-04-03 14:15:35.462794,4189.27,688,refund,San Antonio,0,14,Wednesday,April,2024
1,2,2024-03-19 13:20:35.462824,2659.71,109,refund,Dallas,0,13,Tuesday,March,2024
2,3,2024-01-08 10:08:35.462834,784.00,394,purchase,New York,0,10,Monday,January,2024
3,4,2024-04-13 23:50:35.462850,3514.40,944,purchase,Philadelphia,0,23,Saturday,April,2024
4,5,2024-07-12 18:51:35.462858,369.07,475,purchase,Phoenix,0,18,Friday,July,2024


In [15]:
credit_fraud_new = credit_fraud.copy()

In [16]:
credit_fraud_new 


,TransactionID,TransactionDate,Amount,MerchantID,TransactionType,Location,IsFraud,TransactionHour,TransactionDay,TransactionMonth,TransactionYear
0,1,2024-04-03 14:15:35.462794,4189.27,688,refund,San Antonio,0,14,Wednesday,April,2024
1,2,2024-03-19 13:20:35.462824,2659.71,109,refund,Dallas,0,13,Tuesday,March,2024
2,3,2024-01-08 10:08:35.462834,784.00,394,purchase,New York,0,10,Monday,January,2024
3,4,2024-04-13 23:50:35.462850,3514.40,944,purchase,Philadelphia,0,23,Saturday,April,2024
4,5,2024-07-12 18:51:35.462858,369.07,475,purchase,Phoenix,0,18,Friday,July,2024
...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,2024-06-07 00:57:36.027591,1057.29,289,refund,San Antonio,0,0,Friday,June,2024
99996,99997,2023-10-22 23:12:36.027594,297.25,745,refund,San Antonio,0,23,Sunday,October,2023
99997,99998,2024-05-31 19:27:36.027597,3448.56,690,purchase,San Antonio,0,19,Friday,May,2024
99998,99999,2024-10-18 09:43:36.027601,3750.79,644,purchase,Philadelphia,0,9,Friday,October,2024


MODEL TRAINING

In [17]:
#Split the data into training and testing sets.

x = credit_fraud_new.drop(['IsFraud', 'TransactionDate', 'MerchantID', 'TransactionID'], axis=1)
y = credit_fraud_new['IsFraud']

In [18]:
x.shape, y.shape

((100000, 7), (100000,))

In [19]:
#Train/Test Split the dataset into training and testing sets. We will use 80% of the data for training and 20% for testing.

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

#test size of 0.2 means that 20% of the data will be used for testing and 80% for training. 
#The random_state parameter is set to 42 to ensure that the split is reproducible. 
#The stratify parameter is set to y to ensure that the class distribution in the training and testing sets is similar to that of the original dataset (legitimate=99%, fraudulent=1%).

In [20]:
#Verify the split
x_train.shape, x_test.shape

((80000, 7), (20000, 7))

In [21]:
y_train.shape, y_test.shape

((80000,), (20000,))

In [22]:
#Check the class distribution in the training and testing sets to ensure that they are similar to that of the original dataset.
y_train.value_counts(normalize=True) * 100

IsFraud
0    99.0
1     1.0
Name: proportion, dtype: float64

In [23]:
y_test.value_counts(normalize=True) * 100

IsFraud
0    99.0
1     1.0
Name: proportion, dtype: float64

In [24]:
#Check the sum of fraud counts in the training and testing sets to ensure that they are similar to that of the original dataset.
print("Training fraud cases:", y_train.sum())

Training fraud cases: 800


In [25]:
print("Testing fraud cases:", y_test.sum())

Testing fraud cases: 200


PREPROCESSING

In [26]:
#Here we will decide how each feature will be processed before building the model pipeline. 
#We will use one-hot encoding for categorical variables and standard scaling for numerical variables.

x_train.nunique() 

Amount              73953
TransactionType         2
Location               10
TransactionHour        24
TransactionDay          7
TransactionMonth       12
TransactionYear         2
dtype: int64

In [27]:
numeric_features = ['Amount', 'TransactionHour', 'TransactionYear']
categorical_features = ['TransactionType', 'Location', 'TransactionDay', 'TransactionMonth']

In [28]:
#ColumnTransformer allows us to say: 
#"Apply this transformation to these columns, and another transformation to those columns."
#This is much better than manually encoding columns one by one because it makes the preprocessing reproducible and less prone to leakage.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [29]:
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), numeric_features), ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

In [30]:
#Fit the processor only on the training data.
x_train_processed = preprocessor.fit_transform(x_train)

In [31]:
#Now transform the test data using the same processor. This is important to ensure that the test data is transformed in the same way as the training data.
x_test_processed = preprocessor.transform(x_test)

In [32]:
#Check the results
x_train_processed.shape, x_test_processed.shape

((80000, 34), (20000, 34))

MODEL CREATION 

We will use Logistic regression because:
- it's relatively simple;
- it's fast to train;
- its behavior is comparatively interpretable;
- it gives us a reference point for more complex models.

In [33]:
#LogisticRegression is a classification algorithm designed to estimate the probability that an observation belongs to a particular class.
from sklearn.linear_model import LogisticRegression

In [34]:
logistic_model = LogisticRegression(random_state=42, max_iter=1000)


Logistic Regression uses an iterative optimization process to find its coefficients.

The default number of iterations may sometimes be insufficient, especially after preprocessing several categorical variables.

So:

*max_iter=1000*

gives the optimizer more room to converge.

It doesn't mean the model necessarily performs 1,000 iterations. It means up to 1,000 iterations are allowed.

*random_state=42*

Again, this helps make the experiment reproducible.

In [35]:
#TRAIN THE MODEL
logistic_model.fit(x_train_processed, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [36]:
y_pred = logistic_model.predict(x_test_processed)

In [37]:
y_pred_prob = logistic_model.predict_proba(x_test_processed)[:, 1]     #This gives the probability of each fraud class (1) for each observation in the test set.

In [38]:
print(y_pred[:20])
print(y_pred_prob[:20])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0.00736796 0.00835705 0.00610112 0.01113796 0.00653791 0.00892048
 0.01039081 0.00737931 0.01005557 0.0078273  0.00774851 0.01165984
 0.0169113  0.01231038 0.01286731 0.00816696 0.00993094 0.009238
 0.01211463 0.00767059]


In [39]:
#Save the artifacts and obtained prediction values for evaluation of the model.
import joblib
from pathlib import Path

# Create a folder for saved model artifacts
artifact_dir = Path(".")

# Save the fitted Logistic Regression model
joblib.dump(logistic_model, artifact_dir / "logistic_regression_model.pkl"
)

# Save the fitted preprocessing transformer
joblib.dump(preprocessor, artifact_dir / "fraud_preprocessor.pkl"
)

# Save the processed test features and actual labels
joblib.dump(x_test_processed, artifact_dir / "X_test_processed.pkl"
)

joblib.dump(y_test, artifact_dir / "y_test.pkl"
)

print("Model and evaluation artifacts saved successfully.")

Model and evaluation artifacts saved successfully.
